In [0]:
import csv
from faker import Faker
import random
import os

In [0]:
NUM_RECORDS = 2_000_000  # Adjust as needed, 10GB will require a very large number of records.
FILE_NAME = '/dbfs/y_files/y_emp_details.csv'

In [0]:
fake = Faker('en_US') 

In [0]:
def get_random_department():
    departments = ['HR', 'Marketing', 'Engineering', 'Sales', 'Finance', 'IT', 'Operations']
    return random.choice(departments)

def get_random_job_title(department):
    if department == 'HR':
        return random.choice(['HR Manager', 'Recruiter', 'HR Specialist'])
    elif department == 'Marketing':
        return random.choice(['Marketing Manager', 'Content Creator', 'SEO Specialist'])
    elif department == 'Engineering':
        return random.choice(['Software Engineer', 'DevOps Engineer', 'QA Engineer'])
    # Add more logic for other departments and titles
    return fake.job() # Use Faker's default job if no specific mapping

# --- Main data generation logic ---
def generate_large_employee_data(num_records):
    print(f"Generating {num_records:,} employee records...")
    with open(FILE_NAME, 'w', newline='') as csvfile:
        fieldnames = [
            'Employee_ID', 'First_Name', 'Last_Name', 'Email', 'Phone_Number',
            'Date_of_Birth', 'Gender', 'Address', 'City', 'State', 'Zip_Code',
            'Department', 'Job_Title', 'Hire_Date', 'Salary', 'Employment_Status'
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for i in range(num_records):
            department = get_random_department()
            job_title = get_random_job_title(department)
            
            employee_data = {
                'Employee_ID': i + 1,
                'First_Name': fake.first_name(),
                'Last_Name': fake.last_name(),
                'Email': fake.email(),
                'Phone_Number': fake.phone_number(),
                'Date_of_Birth': fake.date_of_birth(minimum_age=18, maximum_age=65).strftime('%Y-%m-%d'),
                'Gender': random.choice(['Male', 'Female', 'Non-binary']),
                'Address': fake.street_address(),
                'City': fake.city(),
                'State': fake.state_abbr(),
                'Zip_Code': fake.postcode(),
                'Department': department,
                'Job_Title': job_title,
                'Hire_Date': fake.date_between(start_date='-10y', end_date='today').strftime('%Y-%m-%d'),
                'Salary': round(random.uniform(40000, 150000), 2),  # Random salary between 40k and 150k
                'Employment_Status': random.choice(['Full-time', 'Part-time', 'Contractor'])
            }
            writer.writerow(employee_data)
            
            if (i + 1) % 100_000 == 0:
                print(f"Generated {i + 1:,} records...")

    print(f"Finished generating {num_records:,} employee records to {FILE_NAME}")



In [0]:
# --- Execute generation ---
if __name__ == "__main__":
    generate_large_employee_data(NUM_RECORDS)

In [0]:
df= spark.read.csv('dbfs:/y_files/y_emp_details.csv')

In [0]:
df.count()

In [0]:
display(dbutils.fs.ls("dbfs:/y_files/y_emp_details.csv"))

In [0]:
dbutils.fs.mkdirs('/y_files')

# Moving data file to azure adls location

In [0]:
# get details 
var_strg_container = dbutils.secrets.get(scope="y_ss_backed_by_akv", key="yConNam")
var_strg_account   = dbutils.secrets.get(scope="y_ss_backed_by_akv", key="yStrgNam")
var_key            = dbutils.secrets.get(scope="y_ss_backed_by_akv", key="yAk")

In [0]:
# The source URI for your storage container
source_uri = "wasbs://<container-name>@<storage-account-name>.blob.core.windows.net/"

# The mount point path in Databricks
mount_point = "/mnt/<your-mount-point-name>"

# Configuration for the mount
configs = {
    "fs.azure.account.key.<storage-account-name>.blob.core.windows.net":
    dbutils.secrets.get(scope="<secret-scope-name>", key="<access-key-name>")
}

# Mount the storage container
dbutils.fs.mount(
    source=source_uri,
    mount_point=mount_point,
    extra_configs=configs
)

In [0]:
dbutils.secrets.listScopes()